In [10]:
import os
import joblib
import librosa
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

In [11]:
def extract_comprehensive_features(file_path):
    try:
        # Load audio signal
        y, sr = librosa.load(file_path, res_type='kaiser_fast', sr=None)
        
        # 1. Temporal Feature: Zero Crossing Rate
        zcr = librosa.feature.zero_crossing_rate(y).mean()
        
        # 2. Spectral Features: Centroid &amp; Bandwidth
        spec_centroid = librosa.feature.spectral_centroid(y=y, sr=sr).mean()
        spec_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr).mean()
        
        # 3. Tonal &amp; Harmonic Features: Chroma &amp; Tonnetz
        chroma_mean = librosa.feature.chroma_stft(y=y, sr=sr).mean(
            axis=1
        )  # 12 pitch classes
        tonnetz_mean = librosa.feature.tonnetz(y=y, sr=sr).mean(
            axis=1
        )  # 6 harmonic relations
        
        # 4. Cepstral Features: MFCCs (20 coefficients)
        mfccs_mean = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20).mean(axis=1)
        
        # Combine into a single structured 1D feature vector
        feature_vector = np.hstack([
            zcr,
            spec_centroid,
            spec_bandwidth,
            chroma_mean,
            tonnetz_mean,
            mfccs_mean,
        ])
        return feature_vector
    except Exception as e:
        print(f'Error processing {file_path}: {e}')
        return None

In [16]:
def main():
    csv_path = 'UrbanSound_Filtered/filtered_metadata.csv'
    audio_dir = 'UrbanSound_Filtered/audio/'
    
    df = pd.read_csv(csv_path)
    print("CSV Columns:", df.columns.tolist())
    print(f'Loaded {len(df)} metadata rows.')
    
    X, y = [], []
    
    for idx, row in df.iterrows():
        file_path = os.path.join(audio_dir, row['slice_file_name'])
        label = row['class']
    
    features = extract_comprehensive_features(file_path)
    if features is not None:
        X.append(features)
        y.append(label)
    
    X = np.array(X)
    y = np.array(y)
    
    # Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # Train Random Forest Classifier
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, y_train)
    
    # Evaluate
    y_pred = clf.predict(X_test)
    print(f'\nTest Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n')
    print('Classification Report:')
    print(classification_report(y_test, y_pred))
    
    # Save trained model
    joblib.dump(clf, 'audio_classifier.joblib')
    print("Saved multi-feature model to 'audio_classifier.joblib'")

In [17]:
if __name__ == '__main__':
  main()

CSV Columns: ['slice_file_name', 'fsID', 'start', 'end', 'salience', 'fold', 'classID', 'class']
Loaded 3000 metadata rows.


C:\Users\jayog\Documents\INFO4000\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ValueError: With n_samples=1, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.